# YOLO26s — تشخیص انسان در تصاویر هوایی VisDrone
## نسخه نهایی بهینه‌شده برای Google Colab

ویژگی‌های این نسخه:

- مدل رسمی `yolo26s.pt`
- بدون P2 و بدون تغییر معماری
- آموزش تک‌مرحله‌ای
- حالت `quick` برای تست نوت‌بوک و حالت `full` برای آموزش اصلی
- حالت کامل: ۴۵ اپوک و وضوح ۱۲۸۰
- Batch ثابت برای جلوگیری از AutoBatch و خطای OOM
- غیرفعال‌سازی Disk Cache عظیم، Multi-scale و CutMix سنگین
- تبدیل `pedestrian` و `people` به یک کلاس `person`
- ذخیره مستقیم Run، Checkpoint و خروجی‌ها در Google Drive
- Resume پس از قطع‌شدن Colab
- ارزیابی Validation و پشتیبانی از Test اختصاصی کاربر
- Export به PyTorch، TorchScript، ONNX، OpenVINO و NCNN
- ساخت اسکریپت TensorRT برای GPU مقصد

## سلول ۱ — نصب وابستگی‌ها

این سلول PyTorch خود Colab را تغییر نمی‌دهد تا CUDA محیط خراب نشود.

In [1]:
# ============================================================
# CELL 1 — Install dependencies
# ============================================================

import subprocess
import sys

PACKAGES = [
    "ultralytics==8.4.114",
    "albumentations>=2.0.8",
    "opencv-python-headless>=4.10.0",
    "pyyaml>=6.0.2",
    "tqdm>=4.67.0",
    "pandas>=2.2.0",
    "matplotlib>=3.9.0",
    "pillow>=10.4.0",
    "onnx>=1.17.0",
    "onnxruntime>=1.20.0",
    "onnxslim>=0.1.65",
    "openvino>=2025.0.0",
]

command = [
    sys.executable,
    "-m",
    "pip",
    "install",
    "--upgrade",
    "--quiet",
    *PACKAGES,
]

print("Installing required packages...")
subprocess.check_call(command)
print("Package installation completed.")

Installing required packages...
Package installation completed.


## سلول ۲ — اتصال Google Drive و تنظیم پروژه

مقدار `TRAINING_MODE` را روی یکی از گزینه‌های زیر قرار دهید:

- `quick`: تست سریع سلامت نوت‌بوک با ۲ اپوک، وضوح ۶۴۰ و ۵٪ داده
- `full`: آموزش اصلی با ۴۵ اپوک، وضوح ۱۲۸۰ و کل داده

حالت پیش‌فرض روی `full` قرار دارد.

In [2]:
# ============================================================
# CELL 2 — Mount Drive and configure the final project
# ============================================================

from datetime import datetime
from pathlib import Path
import json
import os
import platform
import random
import shutil
import sys

import numpy as np
import torch
import ultralytics
from google.colab import drive
from ultralytics import settings


# ------------------------------------------------------------
# Mount Google Drive
# ------------------------------------------------------------

drive.mount("/content/drive")


# ------------------------------------------------------------
# Training mode
# Options: "quick" or "full"
# ------------------------------------------------------------

TRAINING_MODE = "full"

if TRAINING_MODE == "quick":
    MODEL_NAME = "yolo26s.pt"
    IMAGE_SIZE = 640
    EPOCHS = 2
    BATCH_SIZE = 8
    DATA_FRACTION = 0.05
    WORKERS = 2

    RUN_BASE_NAME = (
        "yolo26s_visdrone_person_quick_test"
    )

    EXPORT_TORCHSCRIPT = False
    EXPORT_ONNX = True
    EXPORT_OPENVINO = False
    EXPORT_NCNN = False
    EXPORT_TENSORRT_IN_COLAB = False
    EXPORT_INT8_VARIANTS = False
    VALIDATE_EXPORTED_MODELS = False
    CREATE_ZIP_BUNDLE = True

elif TRAINING_MODE == "full":
    MODEL_NAME = "yolo26s.pt"
    IMAGE_SIZE = 1280
    EPOCHS = 45
    BATCH_SIZE = 8
    DATA_FRACTION = 1.0
    WORKERS = 4

    RUN_BASE_NAME = (
        "yolo26s_visdrone_person_45e_final"
    )

    EXPORT_TORCHSCRIPT = True
    EXPORT_ONNX = True
    EXPORT_OPENVINO = True
    EXPORT_NCNN = True
    EXPORT_TENSORRT_IN_COLAB = False
    EXPORT_INT8_VARIANTS = False
    VALIDATE_EXPORTED_MODELS = False
    CREATE_ZIP_BUNDLE = True

else:
    raise ValueError(
        "TRAINING_MODE must be either 'quick' or 'full'."
    )


# ------------------------------------------------------------
# Main paths
# ------------------------------------------------------------

SEED = 42

RUN_TIMESTAMP = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

RUN_NAME = (
    f"{RUN_BASE_NAME}_{RUN_TIMESTAMP}"
)

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/"
    "AerialPerson_YOLO26s_Final"
)

RUNS_ROOT = DRIVE_ROOT / "runs"
METADATA_ROOT = DRIVE_ROOT / "metadata"
DEPLOYMENTS_ROOT = DRIVE_ROOT / "deployments"
EVALUATION_ROOT = DRIVE_ROOT / "evaluation"

LOCAL_DATASETS_ROOT = Path(
    "/content/datasets"
)

PERSON_DATASET_ROOT = Path(
    "/content/aerial_person_data/"
    "visdrone_person"
)

RUN_DIR = RUNS_ROOT / RUN_NAME
DEPLOYMENT_DIR = DEPLOYMENTS_ROOT / RUN_NAME


# ------------------------------------------------------------
# Resume configuration
# ------------------------------------------------------------

# Leave empty for a new run.
# For recovery, paste the exact last.pt path from Google Drive.
RESUME_CHECKPOINT = ""

# Rebuild the generated person-only dataset only when needed.
REBUILD_PERSON_DATASET = False


# ------------------------------------------------------------
# Evaluation configuration
# ------------------------------------------------------------

# The private 1000-image test set should have its own YOLO data.yaml.
# Leave empty until that dataset is ready.
CUSTOM_TEST_DATA_YAML = ""

# The official VisDrone test-dev split is diagnostic only.
# Keep False when the final evaluation must use only the private test.
EVALUATE_VISDRONE_TEST = False


# ------------------------------------------------------------
# Reproducibility and speed
# ------------------------------------------------------------

os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Fixed batch size is used, so benchmark mode is safe and faster.
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False


# ------------------------------------------------------------
# Device selection
# ------------------------------------------------------------

CUDA_AVAILABLE = torch.cuda.is_available()

if not CUDA_AVAILABLE:
    raise RuntimeError(
        "CUDA is not available. In Colab select "
        "Runtime > Change runtime type > GPU."
    )

DEVICE = 0
GPU_NAME = torch.cuda.get_device_name(0)


# ------------------------------------------------------------
# Create directories
# ------------------------------------------------------------

for directory in [
    DRIVE_ROOT,
    RUNS_ROOT,
    METADATA_ROOT,
    DEPLOYMENTS_ROOT,
    EVALUATION_ROOT,
    LOCAL_DATASETS_ROOT,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# ------------------------------------------------------------
# Ultralytics paths
# ------------------------------------------------------------

settings.update(
    {
        "datasets_dir": str(
            LOCAL_DATASETS_ROOT
        ),
        "runs_dir": str(
            RUNS_ROOT
        ),
    }
)


# ------------------------------------------------------------
# Save global configuration
# ------------------------------------------------------------

GLOBAL_CONFIG = {
    "created_at": datetime.now().isoformat(),
    "training_mode": TRAINING_MODE,
    "model": MODEL_NAME,
    "image_size": IMAGE_SIZE,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "data_fraction": DATA_FRACTION,
    "workers": WORKERS,
    "seed": SEED,
    "device": DEVICE,
    "gpu_name": GPU_NAME,
    "run_name": RUN_NAME,
    "run_dir": str(RUN_DIR),
    "deployment_dir": str(DEPLOYMENT_DIR),
    "person_dataset_root": str(
        PERSON_DATASET_ROOT
    ),
    "resume_checkpoint": RESUME_CHECKPOINT,
    "custom_test_data_yaml": CUSTOM_TEST_DATA_YAML,
    "evaluate_visdrone_test": EVALUATE_VISDRONE_TEST,
    "ultralytics_version": ultralytics.__version__,
    "torch_version": torch.__version__,
    "cuda_runtime": torch.version.cuda,
    "python_version": sys.version,
    "platform": platform.platform(),
}

config_path = (
    METADATA_ROOT
    / f"{RUN_NAME}_global_config.json"
)

config_path.write_text(
    json.dumps(
        GLOBAL_CONFIG,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


print("=" * 72)
print("FINAL PROJECT CONFIGURATION")
print("=" * 72)
print("Training mode:  ", TRAINING_MODE)
print("Model:          ", MODEL_NAME)
print("Epochs:         ", EPOCHS)
print("Image size:     ", IMAGE_SIZE)
print("Batch size:     ", BATCH_SIZE)
print("Data fraction:  ", DATA_FRACTION)
print("Workers:        ", WORKERS)
print("GPU:            ", GPU_NAME)
print("CUDA runtime:   ", torch.version.cuda)
print("Ultralytics:    ", ultralytics.__version__)
print("Run directory:  ", RUN_DIR)
print("Deployment dir: ", DEPLOYMENT_DIR)
print("Config file:    ", config_path)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Mounted at /content/drive
FINAL PROJECT CONFIGURATION
Training mode:   full
Model:           yolo26s.pt
Epochs:          45
Image size:      1280
Batch size:      8
Data fraction:   1.0
Workers:         4
GPU:             NVIDIA L4
CUDA runtime:    12.8
Ultralytics:     8.4.114
Run directory:   /content/drive/MyDrive/AerialPerson_YOLO26s_Final/runs/yolo26s_visdrone_person_45e_final_20260806_140305
Deployment dir:  /content/drive/MyDrive/AerialPerson_YOLO26s_Final/deployments/yolo26s_visdrone_person_45e_final_20260806_140305
Config file:     /content/drive/MyDrive/AerialPerson_YOLO26s_Final/metadata/yolo26s_visdrone_person_45e_final_20260806_140305_global_config.json


## سلول ۳ — دانلود رسمی VisDrone2019-DET

تنظیم رسمی `VisDrone.yaml` سه بخش Train، Validation و Test-dev را دانلود و Annotationها را به YOLO تبدیل می‌کند.

In [5]:
# ============================================================
# CELL 3 — Download and prepare official VisDrone2019-DET
# ============================================================

from pathlib import Path
import json

from ultralytics.data.utils import check_det_dataset


print("=" * 72)
print("DOWNLOADING / CHECKING OFFICIAL VISDRONE DATASET")
print("=" * 72)

VISDRONE_SOURCE_DATA = check_det_dataset(
    "VisDrone.yaml",
    autodownload=True,
)


def json_safe(value):
    """Convert Path and nested objects to JSON-safe values."""
    if isinstance(value, Path):
        return str(value)

    if isinstance(value, dict):
        return {
            str(key): json_safe(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [
            json_safe(item)
            for item in value
        ]

    return value


source_info_path = (
    METADATA_ROOT
    / f"{RUN_NAME}_visdrone_source.json"
)

source_info_path.write_text(
    json.dumps(
        json_safe(VISDRONE_SOURCE_DATA),
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print("\nResolved VisDrone entries:")

for key in ["path", "train", "val", "test", "names"]:
    print(f"{key:>8}: {VISDRONE_SOURCE_DATA.get(key)}")

print("\nSource metadata:")
print(source_info_path)

DOWNLOADING / CHECKING OFFICIAL VISDRONE DATASET

WARNING ⚠️ Dataset 'VisDrone.yaml' images not found, missing path '/content/datasets/VisDrone/images/val'
Unzipping /content/datasets/VisDrone/VisDrone2019-DET-val.zip to /content/datasets/VisDrone/VisDrone2019-DET-val...: 100% ━━━━━━━━━━━━ 1099/1099 1.2Kfiles/s 0.9s
Unzipping /content/datasets/VisDrone/VisDrone2019-DET-test-dev.zip to /content/datasets/VisDrone/VisDrone2019-DET-test-dev...: 100% ━━━━━━━━━━━━ 3223/3223 1.4Kfiles/s 2.3s
Unzipping /content/datasets/VisDrone/VisDrone2019-DET-train.zip to /content/datasets/VisDrone/VisDrone2019-DET-train...: 100% ━━━━━━━━━━━━ 12945/12945 1.8Kfiles/s 7.3s
Converting train: ━━━━━━━━━━━━ 6471 2.5Kit/s 2.6s
Converting val: ━━━━━━━━━━━━ 548 1.4Kit/s 0.3s
Converting test: ━━━━━━━━━━━━ 1610 2.4Kit/s 0.6s
Dataset download success ✅ (18.1s), saved to /content/datasets


Resolved VisDrone entries:
    path: /content/datasets/VisDrone
   train: /content/datasets/VisDrone/images/train
     val: /conten

## سلول ۴ — ساخت دیتاست Person-only

در VisDrone کلاس‌های `pedestrian` و `people` جدا هستند. این سلول هر دو را به کلاس واحد `person` تبدیل می‌کند و سایر کلاس‌ها را حذف می‌کند.

هیچ Tile یا دیتاست Hybrid ساخته نمی‌شود.

In [6]:
# ============================================================
# CELL 4 — Build person-only VisDrone dataset
# ============================================================

from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple
import hashlib
import json
import os
import shutil

import pandas as pd
import yaml
from tqdm.auto import tqdm


IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".tif",
    ".tiff",
    ".webp",
}


def normalize_names(names) -> Dict[int, str]:
    """Normalize Ultralytics class names to an integer-key dictionary."""
    if isinstance(names, dict):
        return {
            int(key): str(value)
            for key, value in names.items()
        }

    if isinstance(names, (list, tuple)):
        return {
            index: str(value)
            for index, value in enumerate(names)
        }

    raise TypeError(
        f"Unsupported class-name structure: {type(names)}"
    )


def normalize_path_entries(entry) -> List[Path]:
    """Normalize one or many dataset entries to a list of Paths."""
    if entry is None:
        return []

    if isinstance(entry, (list, tuple)):
        raw_entries = list(entry)
    else:
        raw_entries = [entry]

    paths = []

    for raw_entry in raw_entries:
        path = Path(str(raw_entry)).expanduser().resolve()

        if not path.exists():
            raise FileNotFoundError(
                f"Dataset entry does not exist: {path}"
            )

        paths.append(path)

    return paths


def list_images(entry_path: Path) -> List[Path]:
    """Read image paths from a directory or a text-file list."""
    if entry_path.is_dir():
        return sorted(
            path
            for path in entry_path.rglob("*")
            if (
                path.is_file()
                and path.suffix.lower() in IMAGE_EXTENSIONS
            )
        )

    if entry_path.is_file() and entry_path.suffix.lower() == ".txt":
        base_directory = entry_path.parent
        image_paths = []

        for raw_line in entry_path.read_text(
            encoding="utf-8",
            errors="ignore",
        ).splitlines():
            raw_line = raw_line.strip()

            if not raw_line:
                continue

            image_path = Path(raw_line)

            if not image_path.is_absolute():
                image_path = base_directory / image_path

            image_path = image_path.resolve()

            if image_path.exists():
                image_paths.append(image_path)

        return image_paths

    raise ValueError(
        f"Unsupported dataset entry: {entry_path}"
    )


def infer_label_path(image_path: Path) -> Path:
    """Infer the YOLO label path corresponding to an image path."""
    path_parts = list(image_path.parts)

    image_indices = [
        index
        for index, part in enumerate(path_parts)
        if part.lower() == "images"
    ]

    if not image_indices:
        raise ValueError(
            "Could not infer labels directory from image path: "
            f"{image_path}"
        )

    image_index = image_indices[-1]
    path_parts[image_index] = "labels"

    return Path(*path_parts).with_suffix(".txt")


def hardlink_symlink_or_copy(
    source_path: Path,
    destination_path: Path,
) -> None:
    """Create a hard link, then symlink, then copy as a final fallback."""
    destination_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    if destination_path.exists() or destination_path.is_symlink():
        destination_path.unlink()

    try:
        os.link(source_path, destination_path)
        return
    except OSError:
        pass

    try:
        destination_path.symlink_to(source_path.resolve())
        return
    except OSError:
        pass

    shutil.copy2(source_path, destination_path)


def sanitize_person_label_line(
    raw_line: str,
    person_class_ids: Sequence[int],
) -> Optional[str]:
    """Keep pedestrian/people labels and remap both to class 0."""
    values = raw_line.strip().split()

    if len(values) < 5:
        return None

    try:
        class_id = int(float(values[0]))
        x_center = float(values[1])
        y_center = float(values[2])
        box_width = float(values[3])
        box_height = float(values[4])
    except (TypeError, ValueError):
        return None

    if class_id not in person_class_ids:
        return None

    x_center = min(max(x_center, 0.0), 1.0)
    y_center = min(max(y_center, 0.0), 1.0)
    box_width = min(max(box_width, 0.0), 1.0)
    box_height = min(max(box_height, 0.0), 1.0)

    if box_width <= 0.0 or box_height <= 0.0:
        return None

    return (
        "0 "
        f"{x_center:.6f} "
        f"{y_center:.6f} "
        f"{box_width:.6f} "
        f"{box_height:.6f}"
    )


def dataset_is_ready(dataset_root: Path) -> bool:
    """Check whether the generated person-only dataset is reusable."""
    required_paths = [
        dataset_root / "data.yaml",
        dataset_root / "images" / "train",
        dataset_root / "labels" / "train",
        dataset_root / "images" / "val",
        dataset_root / "labels" / "val",
    ]

    return all(path.exists() for path in required_paths)


source_names = normalize_names(
    VISDRONE_SOURCE_DATA["names"]
)

person_class_ids = sorted(
    class_id
    for class_id, class_name in source_names.items()
    if class_name.strip().lower()
    in {
        "pedestrian",
        "people",
        "person",
    }
)

if not person_class_ids:
    raise RuntimeError(
        "The pedestrian/people classes were not found. "
        f"Available names: {source_names}"
    )

print("Source class names:")
print(source_names)
print("Merged person class IDs:")
print(person_class_ids)


if REBUILD_PERSON_DATASET and PERSON_DATASET_ROOT.exists():
    print("Removing the previous generated person dataset...")
    shutil.rmtree(PERSON_DATASET_ROOT)


dataset_statistics = []

if not dataset_is_ready(PERSON_DATASET_ROOT):
    print("\nBuilding person-only dataset...")

    split_mapping = {
        "train": VISDRONE_SOURCE_DATA.get("train"),
        "val": VISDRONE_SOURCE_DATA.get("val"),
        "test": VISDRONE_SOURCE_DATA.get("test"),
    }

    for split_name, split_entry in split_mapping.items():
        if split_entry is None:
            continue

        source_entries = normalize_path_entries(split_entry)

        source_images = []

        for source_entry in source_entries:
            source_images.extend(
                list_images(source_entry)
            )

        source_images = sorted(
            set(source_images)
        )

        destination_images = (
            PERSON_DATASET_ROOT
            / "images"
            / split_name
        )

        destination_labels = (
            PERSON_DATASET_ROOT
            / "labels"
            / split_name
        )

        destination_images.mkdir(
            parents=True,
            exist_ok=True,
        )

        destination_labels.mkdir(
            parents=True,
            exist_ok=True,
        )

        used_names = set()

        image_count = 0
        box_count = 0
        background_count = 0
        missing_label_count = 0
        invalid_line_count = 0

        progress_bar = tqdm(
            source_images,
            desc=f"Creating {split_name}",
            unit="image",
        )

        for image_path in progress_bar:
            destination_name = image_path.name

            if destination_name in used_names:
                short_hash = hashlib.sha1(
                    str(image_path).encode("utf-8")
                ).hexdigest()[:10]

                destination_name = (
                    f"{image_path.stem}_{short_hash}"
                    f"{image_path.suffix.lower()}"
                )

            used_names.add(destination_name)

            destination_image_path = (
                destination_images
                / destination_name
            )

            destination_label_path = (
                destination_labels
                / f"{Path(destination_name).stem}.txt"
            )

            hardlink_symlink_or_copy(
                image_path,
                destination_image_path,
            )

            source_label_path = infer_label_path(
                image_path
            )

            person_lines = []

            if source_label_path.exists():
                raw_lines = source_label_path.read_text(
                    encoding="utf-8",
                    errors="ignore",
                ).splitlines()

                for raw_line in raw_lines:
                    sanitized_line = sanitize_person_label_line(
                        raw_line,
                        person_class_ids,
                    )

                    if sanitized_line is not None:
                        person_lines.append(sanitized_line)
                    elif raw_line.strip():
                        try:
                            raw_class = int(
                                float(
                                    raw_line.strip().split()[0]
                                )
                            )

                            if raw_class in person_class_ids:
                                invalid_line_count += 1
                        except Exception:
                            invalid_line_count += 1
            else:
                missing_label_count += 1

            destination_label_path.write_text(
                "\n".join(person_lines),
                encoding="utf-8",
            )

            image_count += 1
            box_count += len(person_lines)

            if not person_lines:
                background_count += 1

            progress_bar.set_postfix(
                images=image_count,
                boxes=box_count,
                backgrounds=background_count,
            )

        dataset_statistics.append(
            {
                "split": split_name,
                "images": image_count,
                "person_boxes": box_count,
                "background_images": background_count,
                "missing_source_labels": missing_label_count,
                "invalid_person_lines": invalid_line_count,
            }
        )

    PERSON_DATASET_ROOT.mkdir(
        parents=True,
        exist_ok=True,
    )

    data_yaml = {
        "path": str(PERSON_DATASET_ROOT.resolve()),
        "train": "images/train",
        "val": "images/val",
        "names": {
            0: "person",
        },
        "nc": 1,
    }

    if (
        PERSON_DATASET_ROOT
        / "images"
        / "test"
    ).exists():
        data_yaml["test"] = "images/test"

    PERSON_DATA_YAML = (
        PERSON_DATASET_ROOT
        / "data.yaml"
    )

    PERSON_DATA_YAML.write_text(
        yaml.safe_dump(
            data_yaml,
            sort_keys=False,
            allow_unicode=True,
        ),
        encoding="utf-8",
    )

    statistics_frame = pd.DataFrame(
        dataset_statistics
    )

    statistics_path = (
        METADATA_ROOT
        / f"{RUN_NAME}_dataset_statistics.csv"
    )

    statistics_frame.to_csv(
        statistics_path,
        index=False,
    )

else:
    print("\nExisting person-only dataset found and reused.")

    PERSON_DATA_YAML = (
        PERSON_DATASET_ROOT
        / "data.yaml"
    )

    statistics_path = None


if not dataset_is_ready(PERSON_DATASET_ROOT):
    raise RuntimeError(
        "Person-only dataset validation failed."
    )

PERSON_DATA_YAML = str(
    PERSON_DATA_YAML.resolve()
)

print("\n" + "=" * 72)
print("PERSON-ONLY VISDRONE DATASET IS READY")
print("=" * 72)
print("Dataset root:")
print(PERSON_DATASET_ROOT)
print("\nDataset YAML:")
print(PERSON_DATA_YAML)

if dataset_statistics:
    print("\nStatistics:")
    print(pd.DataFrame(dataset_statistics).to_string(index=False))

if statistics_path is not None:
    print("\nStatistics saved to:")
    print(statistics_path)

Source class names:
{0: 'pedestrian', 1: 'people', 2: 'bicycle', 3: 'car', 4: 'van', 5: 'truck', 6: 'tricycle', 7: 'awning-tricycle', 8: 'bus', 9: 'motor'}
Merged person class IDs:
[0, 1]

Building person-only dataset...


Creating train:   0%|          | 0/6471 [00:00<?, ?image/s]

Creating val:   0%|          | 0/548 [00:00<?, ?image/s]

Creating test:   0%|          | 0/1610 [00:00<?, ?image/s]


PERSON-ONLY VISDRONE DATASET IS READY
Dataset root:
/content/aerial_person_data/visdrone_person

Dataset YAML:
/content/aerial_person_data/visdrone_person/data.yaml

Statistics:
split  images  person_boxes  background_images  missing_source_labels  invalid_person_lines
train    6471        106396                787                      0                     0
  val     548         13969                 17                      0                     0
 test    1610         27382                343                      0                     0

Statistics saved to:
/content/drive/MyDrive/AerialPerson_YOLO26s_Final/metadata/yolo26s_visdrone_person_45e_final_20260806_140305_dataset_statistics.csv


## سلول ۵ — بررسی سلامت دیتاست و نمایش نمونه‌ها

این سلول کلاس‌ها، تعداد تصاویر، تعداد باکس‌ها، تصاویر بدون انسان و ابعاد نسبی سوژه‌ها را بررسی می‌کند.

In [7]:
# ============================================================
# CELL 5 — Audit and visualize the person-only dataset
# ============================================================

from pathlib import Path
from typing import Dict, List
import json
import random

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm


def read_yolo_labels(label_path: Path) -> List[List[float]]:
    """Read one YOLO label file."""
    boxes = []

    if not label_path.exists():
        return boxes

    for raw_line in label_path.read_text(
        encoding="utf-8",
        errors="ignore",
    ).splitlines():
        values = raw_line.strip().split()

        if len(values) < 5:
            continue

        try:
            boxes.append(
                [
                    float(values[0]),
                    float(values[1]),
                    float(values[2]),
                    float(values[3]),
                    float(values[4]),
                ]
            )
        except ValueError:
            continue

    return boxes


audit_rows = []

for split_name in ["train", "val", "test"]:
    images_directory = (
        PERSON_DATASET_ROOT
        / "images"
        / split_name
    )

    labels_directory = (
        PERSON_DATASET_ROOT
        / "labels"
        / split_name
    )

    if not images_directory.exists():
        continue

    image_paths = sorted(
        path
        for path in images_directory.iterdir()
        if (
            path.is_file()
            or path.is_symlink()
        )
    )

    box_areas = []
    invalid_class_count = 0
    background_count = 0
    total_boxes = 0

    for image_path in tqdm(
        image_paths,
        desc=f"Auditing {split_name}",
        unit="image",
    ):
        label_path = (
            labels_directory
            / f"{image_path.stem}.txt"
        )

        labels = read_yolo_labels(label_path)

        if not labels:
            background_count += 1

        for class_id, _, _, width, height in labels:
            if int(class_id) != 0:
                invalid_class_count += 1

            box_areas.append(width * height)
            total_boxes += 1

    area_array = np.asarray(
        box_areas,
        dtype=np.float64,
    )

    audit_rows.append(
        {
            "split": split_name,
            "images": len(image_paths),
            "person_boxes": total_boxes,
            "background_images": background_count,
            "invalid_classes": invalid_class_count,
            "median_normalized_area": (
                float(np.median(area_array))
                if area_array.size
                else 0.0
            ),
            "p10_normalized_area": (
                float(np.quantile(area_array, 0.10))
                if area_array.size
                else 0.0
            ),
            "p90_normalized_area": (
                float(np.quantile(area_array, 0.90))
                if area_array.size
                else 0.0
            ),
        }
    )


audit_frame = pd.DataFrame(audit_rows)

if audit_frame.empty:
    raise RuntimeError(
        "No generated dataset splits were found."
    )

if int(audit_frame["invalid_classes"].sum()) != 0:
    raise RuntimeError(
        "The person-only dataset contains invalid class IDs."
    )

audit_path = (
    METADATA_ROOT
    / f"{RUN_NAME}_dataset_audit.csv"
)

audit_frame.to_csv(
    audit_path,
    index=False,
)

print("=" * 72)
print("DATASET AUDIT")
print("=" * 72)
print(audit_frame.to_string(index=False))
print("\nAudit file:")
print(audit_path)


# -----------------------------
# Visualize random train images
# -----------------------------

train_images_directory = (
    PERSON_DATASET_ROOT
    / "images"
    / "train"
)

train_labels_directory = (
    PERSON_DATASET_ROOT
    / "labels"
    / "train"
)

train_image_paths = sorted(
    path
    for path in train_images_directory.iterdir()
    if path.is_file() or path.is_symlink()
)

sample_count = min(6, len(train_image_paths))
sample_paths = random.sample(
    train_image_paths,
    sample_count,
)

figure, axes = plt.subplots(
    2,
    3,
    figsize=(18, 11),
)

axes = np.asarray(axes).reshape(-1)

for axis, image_path in zip(
    axes,
    sample_paths,
):
    image = cv2.imread(str(image_path))

    if image is None:
        axis.set_title("Unreadable image")
        axis.axis("off")
        continue

    image = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2RGB,
    )

    height, width = image.shape[:2]

    label_path = (
        train_labels_directory
        / f"{image_path.stem}.txt"
    )

    labels = read_yolo_labels(label_path)

    for _, x_center, y_center, box_width, box_height in labels:
        x1 = int(
            (x_center - box_width / 2.0)
            * width
        )

        y1 = int(
            (y_center - box_height / 2.0)
            * height
        )

        x2 = int(
            (x_center + box_width / 2.0)
            * width
        )

        y2 = int(
            (y_center + box_height / 2.0)
            * height
        )

        cv2.rectangle(
            image,
            (x1, y1),
            (x2, y2),
            (255, 0, 0),
            2,
        )

    axis.imshow(image)
    axis.set_title(
        f"{image_path.name}\n"
        f"persons={len(labels)}"
    )
    axis.axis("off")

for axis in axes[len(sample_paths):]:
    axis.axis("off")

plt.tight_layout()

preview_path = (
    METADATA_ROOT
    / f"{RUN_NAME}_dataset_preview.jpg"
)

figure.savefig(
    preview_path,
    dpi=160,
    bbox_inches="tight",
)

plt.show()

print("Dataset preview saved to:")
print(preview_path)

Output hidden; open in https://colab.research.google.com to view.

## سلول ۶ — آموزش سریع یا کامل YOLO26s

نسخه کامل با تنظیمات زیر اجرا می‌شود:

- ۴۵ اپوک
- وضوح ثابت ۱۲۸۰
- Batch ثابت ۸
- بدون AutoBatch
- بدون Disk Cache
- بدون Multi-scale
- بدون CutMix
- چهار Worker
- AdamW و Cosine LR
- AMP
- Augmentation متعادل برای تصاویر هوایی

حالت `quick` فقط برای بررسی سالم‌بودن کل Pipeline است.

In [34]:

pip install pathlib

In [35]:
# ============================================================
# CELL 6 — Final stable single-stage YOLO26s training
# ============================================================

from pathlib import Path
import gc
import json
import torch

from ultralytics import YOLO


# ------------------------------------------------------------
# Clean memory before training
# ------------------------------------------------------------

gc.collect()
torch.cuda.empty_cache()

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available. "
        "Select a GPU runtime in Google Colab."
    )


# ------------------------------------------------------------
# Stable training arguments
# ------------------------------------------------------------

TRAIN_ARGUMENTS = {
    # Dataset and duration
    "data": PERSON_DATA_YAML,
    "epochs": EPOCHS,
    "imgsz": IMAGE_SIZE,
    "fraction": DATA_FRACTION,

    # Fixed batch prevents AutoBatch OOM and repeated restarts.
    "batch": BATCH_SIZE,
    "device": DEVICE,
    "workers": WORKERS,

    # Output management
    "project": str(RUNS_ROOT),
    "name": RUN_NAME,
    "exist_ok": False,
    "save": True,
    "save_period": 5,
    "plots": True,
    "verbose": True,

    # Optimization
    "pretrained": True,
    "single_cls": True,
    "optimizer": "AdamW",
    "lr0": 0.00025,
    "lrf": 0.01,
    "weight_decay": 0.0005,
    "warmup_epochs": 1.0,
    "warmup_momentum": 0.8,
    "cos_lr": True,
    "patience": 100,
    "amp": True,
    "compile": False,

    # Stable and fast data pipeline
    "cache": False,
    "multi_scale": False,
    "rect": False,
    "deterministic": False,
    "seed": SEED,

    # Dense aerial scenes
    "max_det": 1000,
    "box": 8.5,
    "cls": 0.30,

    # Lighting and color variation
    "hsv_h": 0.015,
    "hsv_s": 0.40,
    "hsv_v": 0.30,

    # Drone orientation and camera movement
    "degrees": 12.0,
    "translate": 0.08,
    "scale": 0.35,
    "shear": 1.0,
    "perspective": 0.0002,
    "flipud": 0.20,
    "fliplr": 0.50,

    # Small-person and crowded-scene augmentation
    "mosaic": 0.80,
    "mixup": 0.03,
    "cutmix": 0.0,
    "close_mosaic": 5,

    # Validation after every epoch
    "val": True,
}


# ------------------------------------------------------------
# Save exact configuration
# ------------------------------------------------------------

training_config_path = (
    METADATA_ROOT
    / f"{RUN_NAME}_training_arguments.json"
)

training_config_path.write_text(
    json.dumps(
        TRAIN_ARGUMENTS,
        indent=2,
        ensure_ascii=False,
        default=str,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# Start or resume training
# ------------------------------------------------------------

if RESUME_CHECKPOINT:
    resume_path = Path(
        RESUME_CHECKPOINT
    ).expanduser().resolve()

    if not resume_path.exists():
        raise FileNotFoundError(
            f"Resume checkpoint was not found: {resume_path}"
        )

    print("=" * 72)
    print("RESUMING INTERRUPTED TRAINING")
    print("=" * 72)
    print("Checkpoint:", resume_path)

    model = YOLO(
        str(resume_path)
    )

    TRAIN_RESULTS = model.train(
        resume=True
    )

    RUN_DIR = resume_path.parent.parent

else:
    print("=" * 72)
    print("STARTING FINAL SINGLE-STAGE TRAINING")
    print("=" * 72)
    print("Mode:           ", TRAINING_MODE)
    print("Model:          ", MODEL_NAME)
    print("Epochs:         ", EPOCHS)
    print("Image size:     ", IMAGE_SIZE)
    print("Batch size:     ", BATCH_SIZE)
    print("Workers:        ", WORKERS)
    print("Cache:           Disabled")
    print("Multi-scale:     Disabled")
    print("CutMix:          Disabled")
    print("GPU:            ", GPU_NAME)
    print("Output:         ", RUN_DIR)

    model = YOLO(
        MODEL_NAME
    )

    TRAIN_RESULTS = model.train(
        **TRAIN_ARGUMENTS
    )

    RUN_DIR = Path(
        TRAIN_RESULTS.save_dir
    ).resolve()


BEST_PT = (
    RUN_DIR
    / "weights"
    / "best.pt"
)

LAST_PT = (
    RUN_DIR
    / "weights"
    / "last.pt"
)

print("\n" + "=" * 72)
print("TRAINING FINISHED")
print("=" * 72)
print("Run directory:")
print(RUN_DIR)
print("\nBest checkpoint:")
print(BEST_PT)
print("\nLast checkpoint:")
print(LAST_PT)
print("\nTraining configuration:")
print(training_config_path)

ImportError: cannot import name '_Ink' from 'PIL._typing' (/usr/local/lib/python3.12/dist-packages/PIL/_typing.py)

## ادامه آموزش پس از قطع‌شدن Colab

مسیر `last.pt` را در سلول ۲ داخل `RESUME_CHECKPOINT` قرار دهید و سپس سلول‌های ۱، ۲ و ۶ را اجرا کنید.

نمونه:

```python
RESUME_CHECKPOINT = (
    "/content/drive/MyDrive/"
    "AerialPerson_YOLO26s_Final/"
    "runs/نام_اجرا/weights/last.pt"
)
```

Resume همان Run تک‌مرحله‌ای را ادامه می‌دهد.

## سلول ۷ — ارزیابی Validation و Test اختصاصی

- Validation رسمی VisDrone برای انتخاب بهترین مدل استفاده می‌شود.
- Test رسمی VisDrone به‌صورت پیش‌فرض اجرا نمی‌شود.
- پس از آماده‌شدن مجموعه ۱۰۰۰تایی مستقل، مسیر `data.yaml` آن را در `CUSTOM_TEST_DATA_YAML` قرار دهید.

In [9]:
# ============================================================
# CELL 7 — Validation and optional private final test
# ============================================================

from pathlib import Path
import json
import yaml

from ultralytics import YOLO


BEST_PT = (
    RUN_DIR
    / "weights"
    / "best.pt"
)

LAST_PT = (
    RUN_DIR
    / "weights"
    / "last.pt"
)

if not BEST_PT.exists():
    raise FileNotFoundError(
        f"best.pt was not found: {BEST_PT}"
    )

BEST_MODEL = YOLO(
    str(BEST_PT)
)


def metrics_to_dict(metrics):
    """Extract a JSON-safe metric dictionary."""
    output = {}

    if hasattr(metrics, "results_dict"):
        for key, value in metrics.results_dict.items():
            try:
                output[str(key)] = float(value)
            except Exception:
                output[str(key)] = str(value)

    if hasattr(metrics, "speed"):
        output["speed_ms"] = {
            str(key): float(value)
            for key, value in metrics.speed.items()
        }

    return output


evaluation_root = (
    EVALUATION_ROOT
    / RUN_NAME
)

evaluation_root.mkdir(
    parents=True,
    exist_ok=True,
)

evaluation_results = {}


# ------------------------------------------------------------
# Internal validation
# ------------------------------------------------------------

print("=" * 72)
print("VALIDATION SPLIT")
print("=" * 72)

VAL_METRICS = BEST_MODEL.val(
    data=PERSON_DATA_YAML,
    split="val",
    imgsz=IMAGE_SIZE,
    batch=4,
    device=DEVICE,
    conf=0.001,
    max_det=1000,
    plots=True,
    save_json=True,
    project=str(evaluation_root),
    name="validation",
    exist_ok=True,
)

evaluation_results["validation"] = metrics_to_dict(
    VAL_METRICS
)


# ------------------------------------------------------------
# Optional official VisDrone test-dev diagnostic
# ------------------------------------------------------------

person_yaml_content = yaml.safe_load(
    Path(PERSON_DATA_YAML).read_text(
        encoding="utf-8"
    )
)

if (
    EVALUATE_VISDRONE_TEST
    and person_yaml_content.get("test") is not None
):
    print("\n" + "=" * 72)
    print("VISDRONE TEST-DEV DIAGNOSTIC")
    print("=" * 72)

    TEST_METRICS = BEST_MODEL.val(
        data=PERSON_DATA_YAML,
        split="test",
        imgsz=IMAGE_SIZE,
        batch=4,
        device=DEVICE,
        conf=0.001,
        max_det=1000,
        plots=True,
        save_json=True,
        project=str(evaluation_root),
        name="visdrone_test_dev",
        exist_ok=True,
    )

    evaluation_results["visdrone_test_dev"] = (
        metrics_to_dict(TEST_METRICS)
    )


# ------------------------------------------------------------
# Optional private final test
# ------------------------------------------------------------

if CUSTOM_TEST_DATA_YAML:
    private_yaml_path = Path(
        CUSTOM_TEST_DATA_YAML
    ).expanduser().resolve()

    if not private_yaml_path.exists():
        raise FileNotFoundError(
            "Private test data.yaml was not found: "
            f"{private_yaml_path}"
        )

    print("\n" + "=" * 72)
    print("PRIVATE FINAL TEST")
    print("=" * 72)

    PRIVATE_TEST_METRICS = BEST_MODEL.val(
        data=str(private_yaml_path),
        split="test",
        imgsz=IMAGE_SIZE,
        batch=4,
        device=DEVICE,
        conf=0.001,
        max_det=1000,
        plots=True,
        save_json=True,
        project=str(evaluation_root),
        name="private_final_test",
        exist_ok=True,
    )

    evaluation_results["private_final_test"] = (
        metrics_to_dict(PRIVATE_TEST_METRICS)
    )

else:
    print(
        "\nPrivate test skipped because "
        "CUSTOM_TEST_DATA_YAML is empty."
    )


evaluation_json_path = (
    evaluation_root
    / "final_metrics.json"
)

evaluation_json_path.write_text(
    json.dumps(
        evaluation_results,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print("\nFinal metrics:")
print(
    json.dumps(
        evaluation_results,
        indent=2,
        ensure_ascii=False,
    )
)

print("\nMetrics file:")
print(evaluation_json_path)

ImportError: cannot import name '_Ink' from 'PIL._typing' (/usr/local/lib/python3.12/dist-packages/PIL/_typing.py)

## سلول ۸ — جمع‌آوری Checkpointها، گزارش محیط و SHA256

In [10]:
# ============================================================
# CELL 8 — Package PyTorch checkpoints and metadata
# ============================================================

from datetime import datetime
from pathlib import Path
import hashlib
import json
import platform
import shutil
import subprocess
import sys

import torch
import ultralytics


DEPLOYMENT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PYTORCH_DIR = (
    DEPLOYMENT_DIR
    / "pytorch"
)

REPORTS_DIR = (
    DEPLOYMENT_DIR
    / "reports"
)

EXPORTS_DIR = (
    DEPLOYMENT_DIR
    / "exports"
)

SCRIPTS_DIR = (
    DEPLOYMENT_DIR
    / "scripts"
)

for directory in [
    PYTORCH_DIR,
    REPORTS_DIR,
    EXPORTS_DIR,
    SCRIPTS_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


def sha256_file(file_path: Path) -> str:
    """Calculate SHA256 using buffered reads."""
    digest = hashlib.sha256()

    with file_path.open("rb") as file:
        while True:
            chunk = file.read(1024 * 1024)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


checkpoint_records = []

for source_path, destination_name in [
    (BEST_PT, "best.pt"),
    (LAST_PT, "last.pt"),
]:
    if not source_path.exists():
        continue

    destination_path = (
        PYTORCH_DIR
        / destination_name
    )

    shutil.copy2(
        source_path,
        destination_path,
    )

    checkpoint_records.append(
        {
            "name": destination_name,
            "path": str(destination_path),
            "size_bytes": destination_path.stat().st_size,
            "sha256": sha256_file(destination_path),
        }
    )


# Copy important training outputs.
for file_name in [
    "args.yaml",
    "results.csv",
    "results.png",
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "PR_curve.png",
    "P_curve.png",
    "R_curve.png",
    "F1_curve.png",
]:
    source_path = RUN_DIR / file_name

    if source_path.exists():
        shutil.copy2(
            source_path,
            REPORTS_DIR / file_name,
        )


# Save installed package versions.
requirements_path = (
    REPORTS_DIR
    / "requirements_freeze.txt"
)

with requirements_path.open(
    "w",
    encoding="utf-8",
) as file:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "freeze",
        ],
        stdout=file,
        check=True,
        text=True,
    )


environment_report = {
    "created_at": datetime.now().isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "cuda_runtime": torch.version.cuda,
    "cuda_available": torch.cuda.is_available(),
    "gpu": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else None
    ),
    "ultralytics": ultralytics.__version__,
    "model": MODEL_NAME,
    "image_size": IMAGE_SIZE,
    "epochs": EPOCHS,
    "run_dir": str(RUN_DIR),
    "person_data_yaml": PERSON_DATA_YAML,
    "checkpoints": checkpoint_records,
}

environment_path = (
    REPORTS_DIR
    / "environment_and_checkpoints.json"
)

environment_path.write_text(
    json.dumps(
        environment_report,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print("=" * 72)
print("PYTORCH ARTIFACTS")
print("=" * 72)

for record in checkpoint_records:
    print(
        f"{record['name']}: "
        f"{record['size_bytes']:,} bytes"
    )
    print("SHA256:", record["sha256"])

print("\nDeployment directory:")
print(DEPLOYMENT_DIR)

NameError: name 'BEST_PT' is not defined

## سلول ۹ — Export نسخه‌های مختلف مدل

نسخه‌های اصلی:

- `best.pt`: مدل اصلی PyTorch
- TorchScript FP32
- ONNX Legacy Static FP32 برای سازگاری بیشتر با Backend و Tracking
- ONNX Legacy Dynamic FP32 برای اندازه ورودی متغیر
- ONNX End-to-End FP32 برای مسیر سریع YOLO26
- ONNX FP16 در صورت پشتیبانی
- OpenVINO FP32 و FP16
- NCNN
- TensorRT اختیاری

تمام Exportها در `try/except` مستقل هستند؛ خراب‌شدن یک فرمت باعث ازدست‌رفتن فرمت‌های موفق نمی‌شود.

In [11]:
# ============================================================
# CELL 9 — Export deployment formats
# ============================================================

from datetime import datetime
from pathlib import Path
from typing import Any, Dict
import json
import shutil
import traceback

import torch
from tqdm.auto import tqdm
from ultralytics import YOLO


EXPORT_MODEL = YOLO(
    str(BEST_PT)
)


def path_size_bytes(path: Path) -> int:
    """Return file or recursive directory size."""
    if path.is_file():
        return path.stat().st_size

    if path.is_dir():
        return sum(
            file_path.stat().st_size
            for file_path in path.rglob("*")
            if file_path.is_file()
        )

    return 0


def copy_export_artifact(
    source_path: Path,
    destination_path: Path,
) -> Path:
    """Copy one exported file or directory to its final unique name."""
    if destination_path.exists():
        if destination_path.is_dir():
            shutil.rmtree(destination_path)
        else:
            destination_path.unlink()

    if source_path.is_dir():
        shutil.copytree(
            source_path,
            destination_path,
        )
    else:
        destination_path.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        shutil.copy2(
            source_path,
            destination_path,
        )

    return destination_path


def export_with_quantize_fallback(
    export_format: str,
    **kwargs,
):
    """
    Use the current quantize argument and fall back to legacy
    half/int8 arguments for older compatible Ultralytics builds.
    """
    try:
        return EXPORT_MODEL.export(
            format=export_format,
            **kwargs,
        )

    except TypeError:
        quantize = kwargs.pop(
            "quantize",
            None,
        )

        if quantize == 16:
            kwargs["half"] = True

        elif quantize == 8:
            kwargs["int8"] = True

        return EXPORT_MODEL.export(
            format=export_format,
            **kwargs,
        )


export_jobs = []

if EXPORT_TORCHSCRIPT:
    export_jobs.append(
        {
            "label": "torchscript_fp32_static_1280",
            "format": "torchscript",
            "destination": (
                EXPORTS_DIR
                / "yolo26s_person_1280_fp32.torchscript"
            ),
            "kwargs": {
                "imgsz": IMAGE_SIZE,
                "device": DEVICE,
                "optimize": False,
                "end2end": False,
            },
        }
    )


if EXPORT_ONNX:
    export_jobs.extend(
        [
            {
                "label": "onnx_legacy_fp32_static_1280",
                "format": "onnx",
                "destination": (
                    EXPORTS_DIR
                    / "yolo26s_person_legacy_static_1280_fp32.onnx"
                ),
                "kwargs": {
                    "imgsz": IMAGE_SIZE,
                    "device": DEVICE,
                    "dynamic": False,
                    "simplify": True,
                    "opset": 18,
                    "end2end": False,
                },
            },
            {
                "label": "onnx_legacy_fp32_dynamic",
                "format": "onnx",
                "destination": (
                    EXPORTS_DIR
                    / "yolo26s_person_legacy_dynamic_fp32.onnx"
                ),
                "kwargs": {
                    "imgsz": IMAGE_SIZE,
                    "device": DEVICE,
                    "dynamic": True,
                    "simplify": True,
                    "opset": 18,
                    "end2end": False,
                },
            },
            {
                "label": "onnx_end2end_fp32_static_1280",
                "format": "onnx",
                "destination": (
                    EXPORTS_DIR
                    / "yolo26s_person_end2end_static_1280_fp32.onnx"
                ),
                "kwargs": {
                    "imgsz": IMAGE_SIZE,
                    "device": DEVICE,
                    "dynamic": False,
                    "simplify": True,
                    "opset": 18,
                    "end2end": True,
                    "max_det": 1000,
                },
            },
        ]
    )

    if CUDA_AVAILABLE:
        export_jobs.append(
            {
                "label": "onnx_legacy_fp16_static_1280",
                "format": "onnx",
                "destination": (
                    EXPORTS_DIR
                    / "yolo26s_person_legacy_static_1280_fp16.onnx"
                ),
                "kwargs": {
                    "imgsz": IMAGE_SIZE,
                    "device": DEVICE,
                    "dynamic": False,
                    "simplify": True,
                    "opset": 18,
                    "end2end": False,
                    "quantize": 16,
                },
            }
        )

    if EXPORT_INT8_VARIANTS:
        export_jobs.append(
            {
                "label": "onnx_legacy_int8_static_1280",
                "format": "onnx",
                "destination": (
                    EXPORTS_DIR
                    / "yolo26s_person_legacy_static_1280_int8.onnx"
                ),
                "kwargs": {
                    "imgsz": IMAGE_SIZE,
                    "device": DEVICE,
                    "dynamic": False,
                    "simplify": True,
                    "opset": 18,
                    "end2end": False,
                    "quantize": 8,
                    "data": PERSON_DATA_YAML,
                    "fraction": 0.50,
                },
            }
        )


if EXPORT_OPENVINO:
    export_jobs.extend(
        [
            {
                "label": "openvino_fp32_static_1280",
                "format": "openvino",
                "destination": (
                    EXPORTS_DIR
                    / "yolo26s_person_openvino_fp32"
                ),
                "kwargs": {
                    "imgsz": IMAGE_SIZE,
                    "device": "cpu",
                    "dynamic": False,
                    "end2end": False,
                },
            },
            {
                "label": "openvino_fp16_static_1280",
                "format": "openvino",
                "destination": (
                    EXPORTS_DIR
                    / "yolo26s_person_openvino_fp16"
                ),
                "kwargs": {
                    "imgsz": IMAGE_SIZE,
                    "device": "cpu",
                    "dynamic": False,
                    "end2end": False,
                    "quantize": 16,
                },
            },
        ]
    )


if EXPORT_NCNN:
    export_jobs.append(
        {
            "label": "ncnn_fp32_static_1280",
            "format": "ncnn",
            "destination": (
                EXPORTS_DIR
                / "yolo26s_person_ncnn_fp32"
            ),
            "kwargs": {
                "imgsz": IMAGE_SIZE,
                "device": "cpu",
                "end2end": False,
            },
        }
    )


if EXPORT_TENSORRT_IN_COLAB:
    if not CUDA_AVAILABLE:
        print(
            "TensorRT export skipped because CUDA is unavailable."
        )
    else:
        export_jobs.append(
            {
                "label": "tensorrt_fp16_static_1280_colab_gpu",
                "format": "engine",
                "destination": (
                    EXPORTS_DIR
                    / "yolo26s_person_colab_gpu_fp16.engine"
                ),
                "kwargs": {
                    "imgsz": IMAGE_SIZE,
                    "device": 0,
                    "dynamic": False,
                    "workspace": None,
                    "end2end": False,
                    "quantize": 16,
                },
            }
        )

        if EXPORT_INT8_VARIANTS:
            export_jobs.append(
                {
                    "label": "tensorrt_int8_static_1280_colab_gpu",
                    "format": "engine",
                    "destination": (
                        EXPORTS_DIR
                        / "yolo26s_person_colab_gpu_int8.engine"
                    ),
                    "kwargs": {
                        "imgsz": IMAGE_SIZE,
                        "device": 0,
                        "dynamic": False,
                        "workspace": None,
                        "end2end": False,
                        "quantize": 8,
                        "data": PERSON_DATA_YAML,
                        "fraction": 0.50,
                    },
                }
            )


export_records = []

progress_bar = tqdm(
    export_jobs,
    desc="Exporting formats",
    unit="format",
)

for job in progress_bar:
    label = job["label"]

    progress_bar.set_postfix(
        current=label
    )

    record: Dict[str, Any] = {
        "label": label,
        "format": job["format"],
        "requested_at": datetime.now().isoformat(),
        "status": "failed",
        "destination": str(job["destination"]),
        "arguments": {
            key: str(value)
            for key, value in job["kwargs"].items()
        },
    }

    try:
        exported_value = export_with_quantize_fallback(
            job["format"],
            **dict(job["kwargs"]),
        )

        source_path = Path(
            str(exported_value)
        ).resolve()

        if not source_path.exists():
            raise FileNotFoundError(
                "Ultralytics returned an export path that "
                f"does not exist: {source_path}"
            )

        final_path = copy_export_artifact(
            source_path,
            Path(job["destination"]),
        )

        record.update(
            {
                "status": "success",
                "source_path": str(source_path),
                "final_path": str(final_path),
                "size_bytes": path_size_bytes(final_path),
            }
        )

    except Exception as error:
        record.update(
            {
                "error_type": type(error).__name__,
                "error": str(error),
                "traceback": traceback.format_exc(),
            }
        )

        print(
            f"\nExport failed: {label}\n"
            f"{type(error).__name__}: {error}"
        )

    export_records.append(record)


# -----------------------------
# Create app-friendly aliases
# -----------------------------

best_pt_alias = (
    DEPLOYMENT_DIR
    / "best.pt"
)

shutil.copy2(
    PYTORCH_DIR / "best.pt",
    best_pt_alias,
)

legacy_onnx_path = (
    EXPORTS_DIR
    / "yolo26s_person_legacy_static_1280_fp32.onnx"
)

best_onnx_alias = (
    DEPLOYMENT_DIR
    / "best.onnx"
)

if legacy_onnx_path.exists():
    shutil.copy2(
        legacy_onnx_path,
        best_onnx_alias,
    )


export_manifest_path = (
    REPORTS_DIR
    / "export_manifest.json"
)

export_manifest_path.write_text(
    json.dumps(
        export_records,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print("\n" + "=" * 72)
print("EXPORT SUMMARY")
print("=" * 72)

for record in export_records:
    print(
        f"{record['label']}: "
        f"{record['status']}"
    )

    if record["status"] == "success":
        print(
            "  ",
            record["final_path"],
        )
    else:
        print(
            "  ",
            record.get("error"),
        )

print("\nExport manifest:")
print(export_manifest_path)
print("\nApp aliases:")
print(best_pt_alias)

if best_onnx_alias.exists():
    print(best_onnx_alias)

ImportError: cannot import name '_Ink' from 'PIL._typing' (/usr/local/lib/python3.12/dist-packages/PIL/_typing.py)

## سلول ۱۰ — ساخت اسکریپت TensorRT برای RTX 3070

Engine تولیدشده روی GPU کولب ممکن است برای RTX 3070 مناسب یا قابل‌حمل نباشد. این سلول اسکریپت ساخت FP16 و INT8 را کنار مدل ذخیره می‌کند تا روی سیستم مقصد اجرا شود.

In [13]:
# ============================================================
# CELL 10 — Create target-machine TensorRT build script
# ============================================================

from pathlib import Path
import textwrap


tensorrt_script = r"""
from pathlib import Path
import argparse
import json
import shutil

import torch
from ultralytics import YOLO


def main():
    parser = argparse.ArgumentParser(
        description=(
            "Build YOLO26s TensorRT engines on the final NVIDIA GPU."
        )
    )

    parser.add_argument(
        "--model",
        type=Path,
        default=Path("best.pt"),
    )

    parser.add_argument(
        "--data",
        type=str,
        default="",
        help=(
            "Person data.yaml. Required only for INT8 calibration."
        ),
    )

    parser.add_argument(
        "--imgsz",
        type=int,
        default=1280,
    )

    parser.add_argument(
        "--workspace",
        type=float,
        default=4.0,
    )

    parser.add_argument(
        "--build-int8",
        action="store_true",
    )

    args = parser.parse_args()

    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA is unavailable. Install a CUDA-enabled PyTorch build "
            "and a compatible NVIDIA driver."
        )

    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA runtime:", torch.version.cuda)

    model_path = args.model.expanduser().resolve()

    if not model_path.exists():
        raise FileNotFoundError(model_path)

    model = YOLO(str(model_path))

    fp16_result = model.export(
        format="engine",
        imgsz=args.imgsz,
        device=0,
        dynamic=False,
        workspace=args.workspace,
        end2end=False,
        quantize=16,
    )

    fp16_source = Path(str(fp16_result)).resolve()
    fp16_destination = (
        model_path.parent
        / "best_rtx3070_fp16.engine"
    )

    shutil.copy2(
        fp16_source,
        fp16_destination,
    )

    outputs = {
        "fp16": str(fp16_destination),
    }

    if args.build_int8:
        if not args.data:
            raise ValueError(
                "--data is required when --build-int8 is enabled."
            )

        int8_result = model.export(
            format="engine",
            imgsz=args.imgsz,
            device=0,
            dynamic=False,
            workspace=args.workspace,
            end2end=False,
            quantize=8,
            data=args.data,
            fraction=0.50,
        )

        int8_source = Path(str(int8_result)).resolve()
        int8_destination = (
            model_path.parent
            / "best_rtx3070_int8.engine"
        )

        shutil.copy2(
            int8_source,
            int8_destination,
        )

        outputs["int8"] = str(
            int8_destination
        )

    report_path = (
        model_path.parent
        / "tensorrt_build_report.json"
    )

    report_path.write_text(
        json.dumps(
            {
                "gpu": torch.cuda.get_device_name(0),
                "torch": torch.__version__,
                "cuda_runtime": torch.version.cuda,
                "image_size": args.imgsz,
                "outputs": outputs,
            },
            indent=2,
        ),
        encoding="utf-8",
    )

    print(json.dumps(outputs, indent=2))
    print("Report:", report_path)


if __name__ == "__main__":
    main()
"""

tensorrt_script_path = (
    SCRIPTS_DIR
    / "build_tensorrt_on_target.py"
)

tensorrt_script_path.write_text(
    textwrap.dedent(
        tensorrt_script
    ).strip()
    + "\n",
    encoding="utf-8",
)


windows_instructions = r"""
# Run inside the project's Python virtual environment.

python build_tensorrt_on_target.py ^
  --model best.pt ^
  --imgsz 1280 ^
  --workspace 4

# Optional INT8:
# python build_tensorrt_on_target.py ^
#   --model best.pt ^
#   --data path\to\data.yaml ^
#   --imgsz 1280 ^
#   --workspace 4 ^
#   --build-int8
"""

instructions_path = (
    SCRIPTS_DIR
    / "TensorRT_Windows_commands.txt"
)

instructions_path.write_text(
    textwrap.dedent(
        windows_instructions
    ).strip()
    + "\n",
    encoding="utf-8",
)

print("TensorRT target script:")
print(tensorrt_script_path)
print("\nWindows commands:")
print(instructions_path)

TensorRT target script:
/content/drive/MyDrive/AerialPerson_YOLO26s_Final/deployments/yolo26s_visdrone_person_45e_final_20260806_140305/scripts/build_tensorrt_on_target.py

Windows commands:
/content/drive/MyDrive/AerialPerson_YOLO26s_Final/deployments/yolo26s_visdrone_person_45e_final_20260806_140305/scripts/TensorRT_Windows_commands.txt


## سلول ۱۱ — بررسی اختیاری مدل‌های Exportشده

این بخش به‌صورت پیش‌فرض غیرفعال است، زیرا ارزیابی ONNX/OpenVINO می‌تواند زمان‌بر باشد.

In [ ]:
# ============================================================
# CELL 11 — Optional validation of exported models
# ============================================================

from pathlib import Path
import json
import traceback

from ultralytics import YOLO


export_validation_records = []

if not VALIDATE_EXPORTED_MODELS:
    print(
        "Export validation is disabled. "
        "Set VALIDATE_EXPORTED_MODELS=True in Cell 2 to enable it."
    )

else:
    validation_candidates = [
        (
            "pytorch",
            DEPLOYMENT_DIR / "best.pt",
        ),
        (
            "onnx_legacy_fp32",
            DEPLOYMENT_DIR / "best.onnx",
        ),
        (
            "openvino_fp32",
            EXPORTS_DIR / "yolo26s_person_openvino_fp32",
        ),
    ]

    for label, model_path in validation_candidates:
        record = {
            "label": label,
            "path": str(model_path),
            "status": "failed",
        }

        if not model_path.exists():
            record["error"] = "Artifact does not exist."
            export_validation_records.append(record)
            continue

        try:
            deployment_model = YOLO(
                str(model_path)
            )

            deployment_metrics = deployment_model.val(
                data=PERSON_DATA_YAML,
                split="val",
                imgsz=IMAGE_SIZE,
                batch=1,
                conf=0.001,
                max_det=1000,
                device=(
                    DEVICE
                    if label == "pytorch"
                    else "cpu"
                ),
                plots=False,
                verbose=True,
            )

            record["status"] = "success"
            record["metrics"] = metrics_to_dict(
                deployment_metrics
            )

        except Exception as error:
            record["error_type"] = type(error).__name__
            record["error"] = str(error)
            record["traceback"] = traceback.format_exc()

        export_validation_records.append(record)

    validation_path = (
        REPORTS_DIR
        / "export_validation.json"
    )

    validation_path.write_text(
        json.dumps(
            export_validation_records,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    print(
        json.dumps(
            export_validation_records,
            indent=2,
            ensure_ascii=False,
        )
    )

    print("\nExport validation report:")
    print(validation_path)

## سلول ۱۲ — ابزار اختیاری Tiled Inference برای تصاویر هوایی

این تکنیک فقط هنگام استنتاج استفاده می‌شود و معماری یا آموزش مدل را تغییر نمی‌دهد. تصویر به Tileهای هم‌پوشان تقسیم می‌شود، تشخیص‌ها به مختصات تصویر اصلی برگردانده می‌شوند و NMS نهایی اجرا می‌شود.

In [ ]:
# ============================================================
# CELL 12 — Optional tiled inference for tiny aerial persons
# ============================================================

from pathlib import Path
from typing import Dict, List, Tuple
import json

import cv2
import numpy as np
import torch
from torchvision.ops import nms
from tqdm.auto import tqdm
from ultralytics import YOLO


def generate_overlapping_tiles(
    image_width: int,
    image_height: int,
    tile_size: int = 960,
    overlap: float = 0.25,
) -> List[Tuple[int, int, int, int]]:
    """Generate full-coverage overlapping tile coordinates."""
    stride = max(
        1,
        int(
            tile_size
            * (1.0 - overlap)
        ),
    )

    x_starts = list(
        range(
            0,
            max(image_width - tile_size, 0) + 1,
            stride,
        )
    )

    y_starts = list(
        range(
            0,
            max(image_height - tile_size, 0) + 1,
            stride,
        )
    )

    final_x = max(
        image_width - tile_size,
        0,
    )

    final_y = max(
        image_height - tile_size,
        0,
    )

    if not x_starts or x_starts[-1] != final_x:
        x_starts.append(final_x)

    if not y_starts or y_starts[-1] != final_y:
        y_starts.append(final_y)

    tiles = []

    for y1 in sorted(set(y_starts)):
        for x1 in sorted(set(x_starts)):
            x2 = min(
                x1 + tile_size,
                image_width,
            )

            y2 = min(
                y1 + tile_size,
                image_height,
            )

            tiles.append(
                (
                    x1,
                    y1,
                    x2,
                    y2,
                )
            )

    return tiles


def run_tiled_person_inference(
    image_path,
    model_path=BEST_PT,
    output_path=None,
    tile_size: int = 960,
    overlap: float = 0.25,
    confidence: float = 0.10,
    nms_iou: float = 0.55,
    include_full_frame: bool = True,
) -> Dict:
    """Run full-frame plus tiled inference and merge person detections."""
    image_path = Path(
        image_path
    ).expanduser().resolve()

    if not image_path.exists():
        raise FileNotFoundError(
            image_path
        )

    image = cv2.imread(
        str(image_path)
    )

    if image is None:
        raise RuntimeError(
            f"Could not read image: {image_path}"
        )

    image_height, image_width = image.shape[:2]

    inference_model = YOLO(
        str(model_path)
    )

    collected_boxes = []
    collected_scores = []

    def collect_result(
        result,
        offset_x: int,
        offset_y: int,
    ):
        if result.boxes is None:
            return

        xyxy = result.boxes.xyxy.detach().cpu()
        conf = result.boxes.conf.detach().cpu()

        if xyxy.numel() == 0:
            return

        xyxy[:, [0, 2]] += offset_x
        xyxy[:, [1, 3]] += offset_y

        collected_boxes.append(xyxy)
        collected_scores.append(conf)

    if include_full_frame:
        full_result = inference_model.predict(
            source=image,
            imgsz=IMAGE_SIZE,
            conf=confidence,
            classes=[0],
            max_det=1000,
            device=DEVICE,
            verbose=False,
        )[0]

        collect_result(
            full_result,
            0,
            0,
        )

    tile_coordinates = generate_overlapping_tiles(
        image_width=image_width,
        image_height=image_height,
        tile_size=tile_size,
        overlap=overlap,
    )

    for x1, y1, x2, y2 in tqdm(
        tile_coordinates,
        desc="Tiled inference",
        unit="tile",
    ):
        crop = image[
            y1:y2,
            x1:x2,
        ]

        tile_result = inference_model.predict(
            source=crop,
            imgsz=IMAGE_SIZE,
            conf=confidence,
            classes=[0],
            max_det=1000,
            device=DEVICE,
            verbose=False,
        )[0]

        collect_result(
            tile_result,
            x1,
            y1,
        )

    if collected_boxes:
        all_boxes = torch.cat(
            collected_boxes,
            dim=0,
        )

        all_scores = torch.cat(
            collected_scores,
            dim=0,
        )

        keep_indices = nms(
            all_boxes,
            all_scores,
            nms_iou,
        )

        final_boxes = all_boxes[
            keep_indices
        ].numpy()

        final_scores = all_scores[
            keep_indices
        ].numpy()

    else:
        final_boxes = np.empty(
            (0, 4),
            dtype=np.float32,
        )

        final_scores = np.empty(
            (0,),
            dtype=np.float32,
        )

    annotated = image.copy()

    detections = []

    for box, score in zip(
        final_boxes,
        final_scores,
    ):
        x1, y1, x2, y2 = [
            int(round(value))
            for value in box
        ]

        cv2.rectangle(
            annotated,
            (x1, y1),
            (x2, y2),
            (0, 0, 255),
            2,
        )

        cv2.putText(
            annotated,
            f"person {score:.2f}",
            (
                x1,
                max(15, y1 - 5),
            ),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.45,
            (0, 0, 255),
            1,
            cv2.LINE_AA,
        )

        detections.append(
            {
                "xyxy": [
                    x1,
                    y1,
                    x2,
                    y2,
                ],
                "confidence": float(score),
            }
        )

    if output_path is None:
        output_path = (
            image_path.parent
            / f"{image_path.stem}_tiled_detection.jpg"
        )
    else:
        output_path = Path(
            output_path
        ).expanduser().resolve()

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    cv2.imwrite(
        str(output_path),
        annotated,
    )

    json_path = output_path.with_suffix(
        ".json"
    )

    json_path.write_text(
        json.dumps(
            {
                "source": str(image_path),
                "output": str(output_path),
                "tile_size": tile_size,
                "overlap": overlap,
                "confidence": confidence,
                "nms_iou": nms_iou,
                "detections": detections,
            },
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    return {
        "image": str(output_path),
        "json": str(json_path),
        "person_count": len(detections),
    }


# Example:
#
# result = run_tiled_person_inference(
#     image_path="/content/example.jpg",
#     output_path=(
#         DEPLOYMENT_DIR
#         / "examples"
#         / "example_tiled.jpg"
#     ),
# )
#
# print(result)

print(
    "Tiled inference utility is ready. "
    "Uncomment the example after setting an image path."
)

## سلول ۱۳ — Manifest نهایی، هش‌ها و ZIP

In [ ]:
# ============================================================
# CELL 13 — Final manifest, SHA256 list, and ZIP bundle
# ============================================================

from datetime import datetime
from pathlib import Path
import hashlib
import json
import shutil


def collect_artifacts(
    root_directory: Path,
):
    """Collect file metadata recursively."""
    records = []

    for file_path in sorted(
        root_directory.rglob("*")
    ):
        if not file_path.is_file():
            continue

        relative_path = file_path.relative_to(
            root_directory
        )

        records.append(
            {
                "relative_path": str(relative_path),
                "size_bytes": file_path.stat().st_size,
                "sha256": sha256_file(file_path),
            }
        )

    return records


final_artifacts = collect_artifacts(
    DEPLOYMENT_DIR
)

final_manifest = {
    "created_at": datetime.now().isoformat(),
    "project": "Aerial Person YOLO26s Single Stage",
    "model": MODEL_NAME,
    "architecture_modified": False,
    "p2_used": False,
    "hybrid_dataset_used": False,
    "training_stages": 1,
    "epochs": EPOCHS,
    "image_size": IMAGE_SIZE,
    "class_names": {
        "0": "person",
    },
    "source_dataset": "VisDrone2019-DET",
    "person_source_classes": [
        "pedestrian",
        "people",
    ],
    "run_dir": str(RUN_DIR),
    "deployment_dir": str(DEPLOYMENT_DIR),
    "artifacts": final_artifacts,
}

final_manifest_path = (
    DEPLOYMENT_DIR
    / "model_manifest.json"
)

final_manifest_path.write_text(
    json.dumps(
        final_manifest,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


sha256_text_path = (
    DEPLOYMENT_DIR
    / "SHA256SUMS.txt"
)

sha256_text_path.write_text(
    "\n".join(
        f"{record['sha256']}  "
        f"{record['relative_path']}"
        for record in final_artifacts
    )
    + "\n",
    encoding="utf-8",
)


zip_path = None

if CREATE_ZIP_BUNDLE:
    zip_base = (
        DEPLOYMENTS_ROOT
        / f"{RUN_NAME}_deployment_bundle"
    )

    zip_created = shutil.make_archive(
        str(zip_base),
        "zip",
        root_dir=DEPLOYMENT_DIR,
    )

    zip_path = Path(
        zip_created
    )


latest_pointer = {
    "run_name": RUN_NAME,
    "run_dir": str(RUN_DIR),
    "deployment_dir": str(DEPLOYMENT_DIR),
    "best_pt": str(
        DEPLOYMENT_DIR
        / "best.pt"
    ),
    "best_onnx": str(
        DEPLOYMENT_DIR
        / "best.onnx"
    ),
    "manifest": str(
        final_manifest_path
    ),
    "zip_bundle": (
        str(zip_path)
        if zip_path is not None
        else None
    ),
}

latest_pointer_path = (
    DRIVE_ROOT
    / "LATEST_MODEL.json"
)

latest_pointer_path.write_text(
    json.dumps(
        latest_pointer,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


print("=" * 72)
print("PIPELINE COMPLETED")
print("=" * 72)
print("Training run:")
print(RUN_DIR)
print("\nDeployment folder:")
print(DEPLOYMENT_DIR)
print("\nBest PyTorch model:")
print(DEPLOYMENT_DIR / "best.pt")
print("\nBest compatible ONNX alias:")
print(DEPLOYMENT_DIR / "best.onnx")
print("\nManifest:")
print(final_manifest_path)
print("\nSHA256 list:")
print(sha256_text_path)
print("\nLatest-model pointer:")
print(latest_pointer_path)

if zip_path is not None:
    print("\nDeployment ZIP:")
    print(zip_path)

# نکات نهایی

- حالت پیش‌فرض `TRAINING_MODE="full"` است.
- آموزش کامل شامل ۴۵ اپوک، وضوح ۱۲۸۰ و کل داده است.
- حالت `quick` فقط برای تست Pipeline است.
- Batch ثابت ۸ جایگزین AutoBatch شده است.
- Disk Cache، Multi-scale و CutMix سنگین غیرفعال شده‌اند.
- آموزش یک مرحله‌ای و فقط با `yolo26s.pt` انجام می‌شود.
- هیچ P2، Context Tile یا Hybrid Dataset وجود ندارد.
- کلاس‌های `pedestrian` و `people` به `person` تبدیل می‌شوند.
- مدل و Checkpointها مستقیماً در Google Drive ذخیره می‌شوند.
- Test رسمی VisDrone به‌صورت پیش‌فرض برای ارزیابی نهایی اجرا نمی‌شود.
- مجموعه ۱۰۰۰تایی اختصاصی از طریق `CUSTOM_TEST_DATA_YAML` قابل ارزیابی است.
- TensorRT باید روی GPU مقصد، مانند RTX 3070، ساخته شود.